# 02 — Preprocessing & Feature Engineering v3

Feature engineering v3 (36 cech inżynierowanych: 19 v1/v2 + 17 v3) i przygotowanie do modelowania.

**Łącznie po OHE:** 66 kolumn wejściowych (38 numeryczne + 28 z OHE kategorycznych).

> Importuje biblioteki (`pandas`, `sklearn`) oraz moduły projektu potrzebne do preprocessingu i feature engineering.

In [1]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
(ROOT / 'reports').mkdir(parents=True, exist_ok=True)

from config import DROP_COLS, RANDOM_STATE, TARGET, TEST_SIZE
from data_loader import load_hr, save_processed
from features import (
    build_preprocessing_pipeline,
    engineer_features,
    get_X_y,
    get_feature_names,
    save_preprocessed_dataset,
)

## 1. Czyszczenie i nowe cechy

> Ładuje surowe dane HR, generuje 36 cech inżynierowanych (v1/v2 + v3) i wyświetla statystyki opisowe nowych kolumn.

In [2]:
raw = load_hr()
engineered = engineer_features(raw)
print('Poprawka BusinessTravel:', engineered['BusinessTravel'].unique())
new_cols = [c for c in engineered.columns if c not in raw.columns]
print(f'Nowe cechy inżynierowane: {len(new_cols)} (v1/v2 + v3)')
print('v1/v2 (19):', [c for c in new_cols[:19]])
print('v3 (17):', [c for c in new_cols[19:]])
engineered[new_cols].describe().T.round(3)

Poprawka BusinessTravel: <ArrowStringArray>
['Travel_Rarely', 'Travel_Frequently', 'Non-Travel']
Length: 3, dtype: str
Nowe cechy inżynierowane: 36 (v1/v2 + v3)
v1/v2 (19): ['IncomePerYearExp', 'CompanyTenureRatio', 'RoleStabilityRatio', 'PromotionIntensity', 'ManagerTenureRatio', 'CompaniesPerYear', 'HighOvertime', 'AvgSatisfaction', 'LowSatisfaction', 'Stagnation', 'YoungHighMobility', 'IncomeVsDeptMedian', 'CareerStageRatio', 'OvertimeLowSat', 'LongTimeNoPromotion', 'OvertimeHighRiskRole', 'IncomePerJobLevel', 'TenureWithoutPromotion', 'TravelBurden']
v3 (17): ['IncomePerYearAtCompany', 'OvertimeAndLowSatisfaction', 'YoungAndOvertime', 'IncomeVsJobLevel', 'SatisfactionVariance', 'TotalExperienceGap', 'ManagerInstability', 'FrequentJobSwitcher', 'BurnoutRiskScore', 'IncomePerDependent', 'LateCareerNoPromotion', 'CommuteRisk', 'ExperienceMismatch', 'StabilityComposite', 'TravelFatigue', 'LowIncomeHighTenure', 'CareerGrowthIndex']


,count,mean,std,min,25%,50%,75%,max
IncomePerYearExp,1470.0,587.575,284.651,95.286,374.232,549.221,738.304,1904.000
CompanyTenureRatio,1470.0,0.582,0.284,0.000,0.368,0.636,0.833,0.976
RoleStabilityRatio,1470.0,0.481,0.274,0.000,0.333,0.500,0.667,0.882
PromotionIntensity,1470.0,0.236,0.269,0.000,0.000,0.143,0.429,0.917
ManagerTenureRatio,1470.0,0.466,0.277,0.000,0.286,0.500,0.667,0.895
CompaniesPerYear,1470.0,0.279,0.294,0.000,0.091,0.176,0.391,2.250
HighOvertime,1470.0,0.283,0.451,0.000,0.000,0.000,1.000,1.000
AvgSatisfaction,1470.0,2.731,0.506,1.000,2.500,2.750,3.000,4.000
LowSatisfaction,1470.0,0.113,0.317,0.000,0.000,0.000,0.000,1.000
Stagnation,1470.0,0.218,0.413,0.000,0.000,0.000,0.000,1.000


> Zapisuje zbiór z cechami inżynierowanymi do pliku `data/processed/hr_engineered.csv`.

In [3]:
path = save_processed(engineered, 'hr_engineered.csv')
print('Zapisano:', path)

Zapisano: C:\Users\joann\Documents\DataScience\LearnIT\PROJECT\data\processed\hr_engineered.csv


## 2. Podzial X / y i preprocessor

> Buduje macierze X/y, tworzy i dopasowuje pipeline preprocessingu (imputacja mediany → OHE → StandardScaler), wyświetla kształt macierzy i zapisuje `hr_preprocessed.csv`.

In [5]:
X, y, num_cols, cat_cols = get_X_y(raw)
print(f'X: {X.shape}, y: {y.shape}')
print('Usuniete kolumny:', DROP_COLS)
print('Numeryczne:', len(num_cols), '| Kategoryczne:', len(cat_cols))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y,
)

pipeline = build_preprocessing_pipeline(num_cols, cat_cols)
pipeline.fit(X_train, y_train)
X_train_t = pipeline.transform(X_train)
encoder = pipeline.named_steps['encode']
feature_names = get_feature_names(encoder, num_cols, cat_cols)
ohe_names = [n for n in feature_names if n not in num_cols]
print(f'\nPo preprocessingu (train, OHE + StandardScaler): {X_train_t.shape}')
print(f'  numeryczne: {len(num_cols)} | po OneHotEncoding kategorii: {len(ohe_names)}')
print('Kolumny tekstowe (wejscie OHE):', cat_cols)
print('Przyklad kolumn po OHE:', ohe_names[:12])

preprocessed_path = save_preprocessed_dataset(raw)
preprocessed = pd.read_csv(preprocessed_path)
print(f'\nZapisano: {preprocessed_path}')
print(f'Ksztalt (cechy + {TARGET}): {preprocessed.shape}')
preprocessed.head(3)

X: (1470, 66), y: (1470,)
Usuniete kolumny: ['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours']
Numeryczne: 59 | Kategoryczne: 7

Po preprocessingu (train, OHE + StandardScaler): (882, 87)
  numeryczne: 59 | po OneHotEncoding kategorii: 28
Kolumny tekstowe (wejscie OHE): ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Przyklad kolumn po OHE: ['BusinessTravel_Non-Travel', 'BusinessTravel_Travel_Frequently', 'BusinessTravel_Travel_Rarely', 'Department_Human Resources', 'Department_Research & Development', 'Department_Sales', 'EducationField_Human Resources', 'EducationField_Life Sciences', 'EducationField_Marketing', 'EducationField_Medical', 'EducationField_Other', 'EducationField_Technical Degree']

Zapisano: C:\Users\joann\Documents\DataScience\LearnIT\PROJECT\data\processed\hr_preprocessed.csv
Ksztalt (cechy + Attrition): (1470, 88)


,Age,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,...,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Divorced,MaritalStatus_Married,MaritalStatus_Single,OverTime_0,OverTime_1,Attrition
0,0.447645,0.735921,-1.024060,-0.869979,-0.652156,1.447591,0.354705,-0.073426,1.161062,-0.119814,...,-0.245145,-0.486854,1.766775,-0.229142,-0.543281,-0.929889,1.497547,-1.568064,1.568064,1
1,1.318498,-1.329933,-0.164317,-1.837598,0.272606,-0.197995,-1.054528,-0.073426,-0.633954,-0.307097,...,-0.245145,2.054005,-0.566003,-0.229142,-0.543281,1.075397,-0.667759,0.637729,-0.637729,0
2,0.012219,1.416172,-0.901239,-0.869979,1.197368,1.347858,-1.054528,-0.998597,0.263554,-0.966817,...,-0.245145,-0.486854,-0.566003,-0.229142,-0.543281,-0.929889,1.497547,-1.568064,1.568064,1


## 3. Preprocessing i regularizacja

### Preprocessing
- **Numeryczne**: imputacja mediany
- **Kategoryczne**: imputacja moda + `OneHotEncoder`
- **StandardScaler** — po OneHotEncoding; zapis w `hr_preprocessed.csv`
- Usuwamy: `EmployeeCount`, `Over18`, `StandardHours`, `EmployeeNumber`

### Regularizacja (w `src/model.py`)
| Model | Regularizacja |
|-------|----------------|
| Logistic Regression | L2/L1, `C` (0.001–10), solver saga |
| SVM | `C` (0.001–100), RBF/linear |
| MLP | `alpha` 1e-3–1e-1, early stopping |
| XGBoost | `reg_alpha`, `reg_lambda`, płytkie drzewa |
| RF / Balanced RF | `max_depth`, `min_samples_leaf`, `max_features=sqrt` |
| CatBoost | L2 leaf reg, `min_data_in_leaf`, `auto_class_weights='Balanced'` |
| Stacking | L2 w meta-klasyfikatorze (LR) |

### Feature engineering v3 — 36 cech inżynierowanych

**v1/v2 (19 cech):**

| Cecha | Formuła |
|-------|--------|
| `IncomePerYearExp` | `MonthlyIncome / (TotalWorkingYears + 1)` |
| `CompanyTenureRatio` | `YearsAtCompany / (TotalWorkingYears + 1)` |
| `RoleStabilityRatio` | `YearsInCurrentRole / (YearsAtCompany + 1)` |
| `PromotionIntensity` | `YearsSinceLastPromotion / (YearsAtCompany + 1)` |
| `ManagerTenureRatio` | `YearsWithCurrManager / (YearsAtCompany + 1)` |
| `CompaniesPerYear` | `NumCompaniesWorked / (TotalWorkingYears + 1)` |
| `HighOvertime` | `1 jeśli OverTime == 'Yes' else 0` |
| `AvgSatisfaction` | `mean(EnvSat, JobSat, RelSat, WorkLifeBalance)` |
| `LowSatisfaction` | `1 jeśli AvgSatisfaction ≤ 2` |
| `Stagnation` | `1 jeśli YearsSinceLastPromotion ≥ 4 AND YearsAtCompany ≥ 3` |
| `YoungHighMobility` | `1 jeśli Age < 35 AND NumCompaniesWorked ≥ 3` |
| `IncomeVsDeptMedian` | `MonthlyIncome / (mediana dochodu w dziale + 1)` |
| `CareerStageRatio` | `TotalWorkingYears / max(Age − 18, 1)` |
| `OvertimeLowSat` | `1 jeśli OverTime AND AvgSatisfaction ≤ 2.5` |
| `LongTimeNoPromotion` | `1 jeśli YearsSinceLastPromotion ≥ 3` |
| `OvertimeHighRiskRole` | `1 jeśli OverTime AND JobRole ∈ {SalesRep, LabTech}` |
| `IncomePerJobLevel` | `MonthlyIncome / max(JobLevel, 1)` |
| `TenureWithoutPromotion` | `max(YearsAtCompany − YearsSinceLastPromotion, 0)` |
| `TravelBurden` | `DistanceFromHome × TravelWeight (0/1/2)` |

**v3 (17 nowych cech):**

| Cecha | Formuła |
|-------|--------|
| `IncomePerYearAtCompany` | `MonthlyIncome / (YearsAtCompany + 1)` |
| `OvertimeAndLowSatisfaction` | `1 jeśli OverTime AND JobSatisfaction ≤ 2` |
| `YoungAndOvertime` | `1 jeśli Age < 30 AND OverTime` |
| `IncomeVsJobLevel` | `MonthlyIncome / (JobLevel + 1)` |
| `SatisfactionVariance` | `std(EnvSat, JobSat, RelSat, WorkLifeBalance)` |
| `TotalExperienceGap` | `max(TotalWorkingYears − YearsAtCompany, 0)` |
| `ManagerInstability` | `max(YearsAtCompany − YearsWithCurrManager, 0)` |
| `FrequentJobSwitcher` | `1 jeśli NumCompaniesWorked ≥ 5` |
| `BurnoutRiskScore` | `2×HighOvertime + (4 − WorkLifeBalance) + (4 − JobSat)` |
| `IncomePerDependent` | `MonthlyIncome / (NumCompaniesWorked + 1)` |
| `LateCareerNoPromotion` | `1 jeśli Age > 40 AND YearsSinceLastPromotion > 5` |
| `CommuteRisk` | `DistanceFromHome × HighOvertime` |
| `ExperienceMismatch` | `TotalWorkingYears / (JobLevel + 1)` |
| `StabilityComposite` | `YearsWithCurrManager + YearsInCurrentRole + YearsAtCompany` |
| `TravelFatigue` | `TravelWeight × DistanceFromHome × HighOvertime` |
| `LowIncomeHighTenure` | `YearsAtCompany / (MonthlyIncome + 1)` |
| `CareerGrowthIndex` | `JobLevel / (TotalWorkingYears + 1)` |


> Wyświetla statystyki opisowe (mean, std, min, max) dla 15 wybranych cech numerycznych używanych jako wejście do modeli.

In [6]:
# Podglad cech numerycznych uzywanych w modelu
display(X[num_cols].describe().T.head(15))


,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.0,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.0,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.0,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.0,3.0,4.00,5.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.0,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.0,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.0,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.0,2.0,3.00,5.0
JobSatisfaction,1470.0,2.728571,1.102846,1.0,2.0,3.0,4.00,4.0
MonthlyIncome,1470.0,6502.931293,4707.956783,1009.0,2911.0,4919.0,8379.00,19999.0
